In [ ]:
pip install pandas pyfaidx Bio pyarrow fastparquet

In [1]:
import pandas as pd
import subprocess
import os
import sys
import math
from io import StringIO
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

# ==============================
# 1. CONFIGURATIONS
# ==============================
INPUT_PARQUET = r"D:/variant_data/clinvar_clean.parquet"
CHUNK_SIZE = 50000

# Thư mục chứa dữ liệu trên máy host (sẽ được mount vào Docker)
WORKDIR = r"D:/variant_data"
CACHE_DIR = r"D:/vep_cache" 
RESOURCES_DIR = r"D:/vep_resources" # Nơi chứa các file .bw, .vcf.gz cho plugins

ASSEMBLY = "GRCh38"
FASTA_FILENAME = "Homo_sapiens.GRCh38.dna.primary_assembly.fa" 

# Đường dẫn mount trên Docker
DOCKER_WORKDIR = "/input"
DOCKER_CACHE = "/opt/vep/.vep"
DOCKER_RESOURCES = "/opt/vep/resources"

# File tạm
TMP_VCF = os.path.join(WORKDIR, "tmp_input.vcf")
TMP_VEP_OUT = os.path.join(WORKDIR, "tmp_output.txt")

# ==============================
# 2. HELPER FUNCTIONS
# ==============================
def process_chunk(df_chunk, chunk_idx):
    """Xử lý một batch VCF qua VEP Docker và trả về DataFrame đã merge."""
    
    df_chunk["CHROM"] = df_chunk["CHROM"].astype(str).str.replace(r"^chr", "", regex=True, case=False)
    # Tạo ID duy nhất để đảm bảo merge chính xác 100%
    df_chunk["Merge_ID"] = (
        df_chunk["CHROM"].astype(str) + "_" +
        df_chunk["POS"].astype(str) + "_" +
        df_chunk["REF"].astype(str) + "/" +
        df_chunk["ALT"].astype(str)
    )

    # 2.1 Ghi file VCF tạm
    with open(TMP_VCF, "w", newline="") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for _, row in df_chunk.iterrows():
            f.write(f"{row['CHROM']}\t{row['POS']}\t{row['Merge_ID']}\t{row['REF']}\t{row['ALT']}\t.\t.\t.\n")

    # 2.2 Cấu hình lệnh chạy VEP với đầy đủ Plugins
    workdir_mnt = os.path.abspath(WORKDIR).replace('\\', '/')
    cache_mnt = os.path.abspath(CACHE_DIR).replace('\\', '/')
    resources_mnt = os.path.abspath(RESOURCES_DIR).replace('\\', '/')

    vep_cmd = [
        "docker", "run", "--rm",
        "-v", f"{workdir_mnt}:{DOCKER_WORKDIR}",
        "-v", f"{cache_mnt}:{DOCKER_CACHE}",
        "-v", f"{resources_mnt}:{DOCKER_RESOURCES}",
        "ensemblorg/ensembl-vep",
        "vep",
        "-i", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VCF)}",
        "-o", f"{DOCKER_WORKDIR}/{os.path.basename(TMP_VEP_OUT)}",
        "--assembly", ASSEMBLY,
        "--cache", "--offline",
        "--fasta", f"{DOCKER_RESOURCES}/{FASTA_FILENAME}",
        "--tab", "--force_overwrite",
        "--fork", "4",
        
        # Tiêu chí chọn transcript
        "--mane_select", 
        "--canonical",
        "--protein",
        "--hgvs",
        "--pick",
        "--symbol",
        
        # Các cờ thông tin & Tần số quần thể
        "--af_1kg",
        "--af_gnomade",
        "--regulatory",
        
        # Plugins (Đảm bảo file đã có trong thư mục resources)
        "--plugin", f"SpliceAI,snv={DOCKER_RESOURCES}/spliceai_scores.raw.snv.ensembl_mane_v1.4.grch38.vcf.gz",
        "--plugin", (
            f"dbNSFP,{DOCKER_RESOURCES}/dbNSFP5.3.1a_grch38.gz," 
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        ),
        
        # Chỉ định các cột đầu ra (Thêm các trường từ Plugin)
        "--fields", (
            "Uploaded_variation,Location,Allele,Gene,SYMBOL,Feature,Feature_type,Consequence,"
            "cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,"
            "MANE_SELECT,CANONICAL,AF,gnomADe_AF,SpliceAI_pred,"
            "phyloP100way_vertebrate,phyloP470way_mammalian,phyloP17way_primate,"
            "phastCons100way_vertebrate,phastCons470way_mammalian,phastCons17way_primate,"
            "GERP++_RS,GERP++_NR,GERP_92_mammals"
        )
    ]

    # 2.3 Thực thi Docker
    try:
        subprocess.run(vep_cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[LỖI VEP - Chunk {chunk_idx}]")
        print(e.stderr.strip() or "<empty>")
        raise SystemExit("Dừng pipeline do lỗi chạy VEP.")

    # 2.4 Parse kết quả và Merge
    with open(TMP_VEP_OUT) as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith("##")]
    
    header_line = [l for l in lines if l.startswith("#Uploaded_variation")][0]
    cols = header_line.lstrip("#").split("\t")
    data_lines = [l for l in lines if not l.startswith("#")]
    
    vep_df = pd.read_csv(StringIO("\n".join(data_lines)), sep="\t", names=cols)
    vep_df = vep_df.dropna(subset=["Uploaded_variation"])
    
    # Merge lại với chunk ban đầu (Sử dụng Uploaded_variation từ VEP và Merge_ID của df)
    merged_chunk = df_chunk.merge(
        vep_df, 
        left_on="Merge_ID", 
        right_on="Uploaded_variation", 
        how="left"
    )
    
    # Xóa cột ID tạm
    merged_chunk = merged_chunk.drop(columns=["Merge_ID", "Uploaded_variation"], errors="ignore")

    if 'SpliceAI_pred' in merged_chunk.columns:
        # Tách chuỗi bằng dấu '|' (sẽ tạo ra list gồm 9 phần tử)
        split_sa = merged_chunk['SpliceAI_pred'].str.split('|', expand=True)
        
        # Lấy đúng 4 cột điểm Delta Score (Vị trí index 1, 2, 3, 4)
        if split_sa.shape[1] == 9:
            merged_chunk['SpliceAI_pred_DS_AG'] = pd.to_numeric(split_sa[1], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_AL'] = pd.to_numeric(split_sa[2], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DG'] = pd.to_numeric(split_sa[3], errors='coerce')
            merged_chunk['SpliceAI_pred_DS_DL'] = pd.to_numeric(split_sa[4], errors='coerce')
            
    return merged_chunk

In [2]:
print(f"[*] Đang tải dữ liệu từ {INPUT_PARQUET}...")
df_full = pd.read_parquet(INPUT_PARQUET)
total_rows = len(df_full)
total_chunks = math.ceil(total_rows / CHUNK_SIZE)
print(f"[*] Tổng số variant: {total_rows:,}. Số lượng chunks: {total_chunks}")

# Kiểm tra cột bắt buộc
required_cols = {"CHROM", "POS", "REF", "ALT"}
if not required_cols.issubset(df_full.columns):
    raise ValueError(f"File Parquet thiếu các cột: {required_cols - set(df_full.columns)}")

[*] Đang tải dữ liệu từ D:/variant_data/clinvar_clean.parquet...
[*] Tổng số variant: 4,137,591. Số lượng chunks: 83


In [3]:
# Định nghĩa file Parquet kết quả cuối cùng
FINAL_OUTPUT_PARQUET = r"D:/variant_data/clinvar_vep_mapping.parquet"

# Xóa file cũ nếu có để tránh ghi đè lỗi
if os.path.exists(FINAL_OUTPUT_PARQUET): 
    os.remove(FINAL_OUTPUT_PARQUET)

# Khởi tạo đối tượng Writer của PyArrow
parquet_writer = None

print("[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...")

# Xử lý theo vòng lặp
for i in range(total_chunks):
    start_idx = i * CHUNK_SIZE
    end_idx = min((i + 1) * CHUNK_SIZE, total_rows)
    
    print(f"    -> Đang xử lý Chunk {i+1}/{total_chunks} (Rows: {start_idx} - {end_idx})...")
    
    # Cắt batch
    current_chunk = df_full.iloc[start_idx:end_idx].copy()
    
    # Chạy qua VEP Docker
    annotated_chunk = process_chunk(current_chunk, i+1)
    
    # --- CƠ CHẾ GHI CUỐN CHIẾU VÀO PARQUET ---
    # 1. Chuyển đổi Pandas DataFrame thành PyArrow Table
    table = pa.Table.from_pandas(annotated_chunk)
    
    # 2. Khởi tạo Writer ở chunk đầu tiên (để cố định Schema/kiểu dữ liệu của các cột)
    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(FINAL_OUTPUT_PARQUET, table.schema, compression='snappy')
    
    # 3. Ghi trực tiếp table của chunk này xuống ổ cứng
    parquet_writer.write_table(table)
    
    # Giải phóng bộ nhớ RAM ngay lập tức
    del annotated_chunk
    del current_chunk

# BẮT BUỘC: Đóng writer sau khi chạy xong tất cả các chunk để đóng file Parquet chuẩn định dạng
if parquet_writer:
    parquet_writer.close()

# Dọn dẹp các file VCF/TXT tạm thời trên ổ cứng
if os.path.exists(TMP_VCF): os.remove(TMP_VCF)
if os.path.exists(TMP_VEP_OUT): os.remove(TMP_VEP_OUT)

print(f"\n[SUCCESS] Pipeline hoàn tất! Kết quả cuối cùng lưu tại: {FINAL_OUTPUT_PARQUET}")

[*] Bắt đầu chạy pipeline cuốn chiếu xuất ra Parquet...
    -> Đang xử lý Chunk 1/83 (Rows: 0 - 50000)...
    -> Đang xử lý Chunk 2/83 (Rows: 50000 - 100000)...
    -> Đang xử lý Chunk 3/83 (Rows: 100000 - 150000)...
    -> Đang xử lý Chunk 4/83 (Rows: 150000 - 200000)...
    -> Đang xử lý Chunk 5/83 (Rows: 200000 - 250000)...
    -> Đang xử lý Chunk 6/83 (Rows: 250000 - 300000)...
    -> Đang xử lý Chunk 7/83 (Rows: 300000 - 350000)...
    -> Đang xử lý Chunk 8/83 (Rows: 350000 - 400000)...
    -> Đang xử lý Chunk 9/83 (Rows: 400000 - 450000)...
    -> Đang xử lý Chunk 10/83 (Rows: 450000 - 500000)...
    -> Đang xử lý Chunk 11/83 (Rows: 500000 - 550000)...
    -> Đang xử lý Chunk 12/83 (Rows: 550000 - 600000)...
    -> Đang xử lý Chunk 13/83 (Rows: 600000 - 650000)...
    -> Đang xử lý Chunk 14/83 (Rows: 650000 - 700000)...
    -> Đang xử lý Chunk 15/83 (Rows: 700000 - 750000)...
    -> Đang xử lý Chunk 16/83 (Rows: 750000 - 800000)...
    -> Đang xử lý Chunk 17/83 (Rows: 800000 - 85

In [4]:
final_df = pd.read_parquet(FINAL_OUTPUT_PARQUET)

In [5]:
final_df['Consequence'].value_counts()

Consequence
missense_variant                                                                                          2270106
synonymous_variant                                                                                         691731
intron_variant                                                                                             336913
downstream_gene_variant                                                                                    142198
splice_polypyrimidine_tract_variant,intron_variant                                                          97051
upstream_gene_variant                                                                                       93804
stop_gained                                                                                                 83572
missense_variant,splice_region_variant                                                                      75532
splice_region_variant,splice_polypyrimidine_tract_variant,intron_variant    

In [6]:
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                        748
HGNC_ID                          4484
ClinicalSignificance           235824
ClinSigSimple                       0
PhenotypeIDS                    48385
PhenotypeList                     562
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                   235824
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         4136138
ReviewStatusClinicalImpact    4136138
Oncogenicity                  4135773
ReviewStatusOncogenicity      4135773
Location    

In [7]:
import numpy as np

# Danh sách giá trị cần kiểm tra
invalid_values = ["-", "na", "n/a", "NA", "N/A", "NaN", "nan"]

# Thay thế toàn bộ giá trị lỗi trong toàn DataFrame bằng np.nan
final_df = final_df.replace(invalid_values, np.nan)
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                        748
HGNC_ID                          4484
ClinicalSignificance           235824
ClinSigSimple                       0
PhenotypeIDS                    48385
PhenotypeList                     562
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                   235824
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         4136138
ReviewStatusClinicalImpact    4136138
Oncogenicity                  4135773
ReviewStatusOncogenicity      4135773
Location    

In [8]:
final_df = final_df[final_df['Consequence'] == 'missense_variant']
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                          3
HGNC_ID                          1158
ClinicalSignificance            41168
ClinSigSimple                       0
PhenotypeIDS                    18992
PhenotypeList                     364
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                    41168
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         2269037
ReviewStatusClinicalImpact    2269037
Oncogenicity                  2269082
ReviewStatusOncogenicity      2269082
Location    

In [9]:
final_df = final_df.dropna(subset=['ClinicalSignificance', 'Consequence', 'SYMBOL', 'MANE_SELECT', 'CANONICAL'])
final_df.isnull().sum()

#AlleleID                           0
Type                                0
Name                                0
GeneID                              0
GeneSymbol                          0
HGNC_ID                          1130
ClinicalSignificance                0
ClinSigSimple                       0
PhenotypeIDS                    18751
PhenotypeList                     357
Origin                              0
OriginSimple                        0
Assembly                            0
ChromosomeAccession                 0
CHROM                               0
Start                               0
Stop                                0
ReviewStatus                        0
VariationID                         0
POS                                 0
REF                                 0
ALT                                 0
SomaticClinicalImpact         2223203
ReviewStatusClinicalImpact    2223203
Oncogenicity                  2223230
ReviewStatusOncogenicity      2223230
Location    

In [ ]:
final_df.to_parquet(r'D:/variant_data/clinvar_vep_mapping_final.parquet', index=False)